In [ ]:
import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Please enable GPU in Colab.")

GPU Available: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers datasets accelerate matplotlib tqdm sentencepiece huggingface_hub

In [ ]:
%cd /content

!git clone https://github.com/musfiraumar/RLHF-Summarizer-from-scratch.git

%cd /content/RLHF-Summarizer-from-scratch

/content
fatal: destination path 'RLHF-Summarizer-from-scratch' already exists and is not an empty directory.
/content/RLHF-Summarizer-from-scratch


In [ ]:
!ls
!ls starter

common.py	__pycache__  starter	     train_ppo.py
evaluate.py	README.md    test-common.py  train_reward.py
notebook.ipynb	results      train_dpo.py    train_sft.py
batching.py  check_starter.py  config.py  data.py  models.py  __pycache__


In [ ]:
!python starter/check_starter.py

config.json: 100% 704/704 [00:00<00:00, 3.03MB/s]
tokenizer_config.json: 100% 3.66k/3.66k [00:00<00:00, 12.8MB/s]
vocab.json: 100% 801k/801k [00:00<00:00, 29.1MB/s]
merges.txt: 100% 466k/466k [00:00<00:00, 105MB/s]
special_tokens_map.json: 100% 831/831 [00:00<00:00, 5.27MB/s]
tokenizer.json: 100% 2.10M/2.10M [00:00<00:00, 140MB/s]
README.md: 100% 532/532 [00:00<00:00, 3.34MB/s]

data/train-00000-of-00001-e8c59e5cf7bce1(…): downloading bytes:   0% 0.00/111M [00:00<?, ?B/s]
data/train-00000-of-00001-e8c59e5cf7bce1(…): downloading bytes:  25% 27.9M/111M [00:01<00:02, 33.7MB/s, 1.55MB/s  ]
data/train-00000-of-00001-e8c59e5cf7bce1(…): reconstructing file:   9% 10.4M/111M [00:01<00:11, 8.54MB/s, 79.6kB/s  ]
data/train-00000-of-00001-e8c59e5cf7bce1(…): downloading bytes:  35% 38.2M/111M [00:01<00:01, 39.5MB/s, 3.14MB/s  ]
data/train-00000-of-00001-e8c59e5cf7bce1(…): reconstructing file:  16% 18.0M/111M [00:01<00:06, 15.1MB/s, 1.36MB/s  ]
data/train-00000-of-00001-e8c59e5cf7bce1(…): downloadin

In [ ]:
!python train_sft.py

Device: cuda

model.safetensors: downloading bytes:  42% 112M/269M [00:00<00:00, 214MB/s, 7.33MB/s  ] 
model.safetensors: downloading bytes:  57% 154M/269M [00:00<00:00, 208MB/s, 13.3MB/s  ]
model.safetensors: downloading bytes:  69% 186M/269M [00:01<00:00, 186MB/s, 16.1MB/s  ]
model.safetensors: downloading bytes:  77% 208M/269M [00:01<00:00, 194MB/s, 17.5MB/s  ]
model.safetensors: reconstructing file:  87% 234M/269M [00:01<00:00, 201MB/s, 18.9MB/s  ]
model.safetensors: downloading bytes: 100% 233M/233M [00:01<00:00, 160MB/s, 21.5MB/s  ]
model.safetensors: reconstructing file: 100% 269M/269M [00:01<00:00, 185MB/s, 25.3MB/s  ]
Loading weights: 100% 272/272 [00:00<00:00, 4782.88it/s]
generation_config.json: 100% 111/111 [00:00<00:00, 645kB/s]
Baseline validation loss: 2.798371895328208
  4% 49/1250 [03:20<1:24:05,  4.20s/it]Step 50/1250 | Train Loss: 2.6283
  8% 99/1250 [06:46<1:18:03,  4.07s/it]Step 100/1250 | Train Loss: 2.5097
 12% 149/1250 [10:13<1:14:08,  4.04s/it]Step 150/1250 | T

In [ ]:
import json

with open("results/sft_log.json") as f:
    logs = json.load(f)

print("Baseline Val Loss:", logs["baseline_val_loss"])
print("Final Val Loss:", logs["final_val_loss"])
print("Training Time:", logs["total_training_time_seconds"] / 60, "minutes")
print("GPU:", logs["gpu"])

Baseline Val Loss: 2.798371895328208
Final Val Loss: 2.444736842105263
Training Time: 89.51681604385377 minutes
GPU: Tesla T4


In [ ]:
!pip install -q -U huggingface_hub

from huggingface_hub import notebook_login

notebook_login()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


KeyboardInterrupt: 

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="checkpoints/sft",
    repo_id="Musfirah-umar/Rlhf-summarizer",
    repo_type="model"
)

print("SFT model uploaded successfully!")

SFT model uploaded successfully!


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Musfirah-umar/Rlhf-summarizer",
    repo_type="model",
    local_dir="checkpoints/sft",
    token=True
)

LocalTokenNotFoundError: Token is required (`token=True`), but no token found. You need to provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

In [ ]:
!ls -lh checkpoints/sft/

total 260M
-rw-r--r-- 1 root root  795 Nov 12 17:36 config.json
-rw-r--r-- 1 root root  111 Nov 12 17:36 generation_config.json
-rw-r--r-- 1 root root 257M Nov 12 17:36 model.safetensors
-rw-r--r-- 1 root root  775 Nov 12 17:36 tokenizer_config.json
-rw-r--r-- 1 root root 3.4M Nov 12 17:36 tokenizer.json


In [ ]:
%cd /content/RLHF-Summarizer-from-scratch
!git pull

/content/RLHF-Summarizer-from-scratch
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 10 (delta 6), reused 10 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 35.77 KiB | 4.47 MiB/s, done.
From https://github.com/musfiraumar/RLHF-Summarizer-from-scratch
   7e8074b..1aabd14  main       -> origin/main
Updating 7e8074b..1aabd14
Fast-forward
 evaluate.py               |  265 +++++
 notebook.ipynb            | 2509 ++++++++++++++++++++++---------------------
 results/ppo_log_fp32.json | 2602 +++++++++++++++++++++++++++++++++++++++++++++
 train_dpo.py              |    1 -
 4 files changed, 4144 insertions(+), 1233 deletions(-)
 create mode 100644 results/ppo_log_fp32.json


In [ ]:
!python -u train_reward.py

Device: cuda
Training pairs: 20000
Validation pairs: 1000
Loading weights: 100% 272/272 [00:00<00:00, 4663.61it/s]
Baseline validation accuracy: 0.497
  0% 5/2500 [00:11<1:31:31,  2.20s/it][W1112 17:43:26.176779216 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 65011712 bytes (free: 6094848, total: 15637086208).
  1% 16/2500 [00:37<1:35:03,  2.30s/it][W1112 17:43:52.073397782 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 69206016 bytes (free: 50135040, total: 15637086208).
  2% 49/2500 [02:01<1:43:43,  2.54s/it]Step 50 | Train Loss: 0.6929
  4% 99/2500 [04:08<1:41:25,  2.53s/it]Step 100 | Train Loss: 0.6931
  6% 149/2500 [06:17<1:42:48,  2.62s/it]Step 150 | Train Loss: 0.6925
  8% 199/2500 [08:29<1:43:11,  2.69s/it]Step 200 | Train Loss: 0.6917
 10% 249/2500 [10:42<1:39:07,  2.64s/it]Step 250 | Train Loss: 0.6933
Step 250 | Train Loss: 0.6933 | Validation Accuracy: 0.5150
 12%

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="checkpoints/reward",
    repo_id="Musfirah-umar/Rlhf-summarizer",
    repo_type="model",
    path_in_repo="reward"
)

print("Reward model uploaded successfully!")

Reward model uploaded successfully!


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Musfirah-umar/Rlhf-summarizer",
    local_dir="checkpoints/sft",
    allow_patterns=["config.json", "generation_config.json",
                    "model.safetensors", "tokenizer.json",
                    "tokenizer_config.json"],
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

'/content/RLHF-Summarizer-from-scratch/checkpoints/sft'

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Musfirah-umar/Rlhf-summarizer",
    local_dir="checkpoints",
    allow_patterns=["reward/*"],
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'/content/RLHF-Summarizer-from-scratch/checkpoints'

In [ ]:
!ls checkpoints/sft
!ls checkpoints/reward

config.json		model.safetensors      tokenizer.json
generation_config.json	tokenizer_config.json
ls: cannot access 'checkpoints/reward': No such file or directory


In [ ]:
from huggingface_hub import HfApi
for f in HfApi().list_repo_files("Musfirah-umar/Rlhf-summarizer"):
    print(f)

.gitattributes
config.json
generation_config.json
model.safetensors
reward/reward_model.pt
reward/tokenizer.json
reward/tokenizer_config.json
tokenizer.json
tokenizer_config.json


In [ ]:
!python train_ppo.py


Loading weights: 100% 272/272 [00:00<00:00, 4397.24it/s]
Loading weights: 100% 272/272 [00:00<00:00, 4820.00it/s]
Loading weights: 100% 272/272 [00:00<00:00, 4410.01it/s]
Device: cuda
Policy dtype: torch.float32
Value head dtype: torch.float32
Reference dtype: torch.float32
Reward model dtype: torch.float32
Gradient checkpointing: True
[1/200] reward=-0.597 kl=0.000 length=35.6 clip=0.169 epoch0_clip=0.000 ratio_dev=2.23e-04 policy_loss=-0.0621 value_loss=0.4570 entropy=2.744 (0.3 min)
[2/200] reward=-0.570 kl=-0.035 length=36.9 clip=0.052 epoch0_clip=0.000 ratio_dev=2.75e-04 policy_loss=-0.0342 value_loss=0.3776 entropy=2.614 (0.6 min)
[3/200] reward=-0.597 kl=0.100 length=32.2 clip=0.040 epoch0_clip=0.000 ratio_dev=1.74e-04 policy_loss=-0.0325 value_loss=0.3898 entropy=2.916 (0.8 min)
[4/200] reward=-0.536 kl=0.150 length=38.6 clip=0.022 epoch0_clip=0.000 ratio_dev=1.95e-04 policy_loss=-0.0263 value_loss=0.3288 entropy=3.047 (1.2 min)
[5/200] reward=-0.519 kl=0.695 length=33.8 clip=0

In [ ]:
!zip -r ppo_checkpoint.zip checkpoints/ppo

  adding: checkpoints/ppo/ (stored 0%)
  adding: checkpoints/ppo/tokenizer_config.json (deflated 61%)
  adding: checkpoints/ppo/config.json (deflated 50%)
  adding: checkpoints/ppo/model.safetensors (deflated 7%)
  adding: checkpoints/ppo/tokenizer.json (deflated 82%)
  adding: checkpoints/ppo/generation_config.json (deflated 23%)


In [ ]:
from google.colab import files

files.download("ppo_checkpoint.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python train_dpo.py

In [ ]:
!pip install -U huggingface_hub rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 8.9 MB/s  0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24985 sha256=1b78564b39d5fca51ffc9dee2e049d59afa63a12516b71630e065e3b7df8cc64
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [huggingface_hub]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but 

In [ ]:
%cd /content/RLHF-Summarizer-from-scratch


/content/RLHF-Summarizer-from-scratch


In [ ]:
!python evaluate.py

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/269M [00:00<?, ?B/s]          Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/269M [00:00<?, ?B/s]

Fetching 5 files:  20% 1/5 [00:00<00:01,  2.97it/s]
Reconstructing (incomplete total...):   0% 111/269M [00:00<225:54:06, 331B/s]
Reconstructing (incomplete total...):   0% 906/269M [00:00<225:54:06, 331B/s]
Reconstructing (incomplete total...):   0% 1.68k/273M [00:00<228:51:32, 331B/s]

Reconstructing (incomplete total...): 100% 273M/273M [00:02<00:00, 113MB/s, 43.5MB/s  ]

Fetching 5 files: 100% 5/5 [00:02<00:00,  1.68it/s]
Download complete: 100% 236M/236M [00:02<00:00, 187MB/s,  187MB/s  ]
Reconstruction complete: 100% 273M/273M [00:02<00:00, 113MB/s,  113MB/s  ]             




In [ ]:
!ls -l evaluate.py

-rw-r--r-- 1 root root 0 Nov 13 15:10 evaluate.py


In [1]:
"""Build the blinded human-evaluation sheet (Section 3.2)."""
import csv, json, random

SEED, N = 42, 30
MODELS = ["sft", "ppo", "dpo"]

rows = [json.loads(l) for l in open("results/eval_summaries.jsonl", encoding="utf-8")]
rng = random.Random(SEED)

# keep only posts where the three summaries differ
pool = [i for i, r in enumerate(rows)
        if len({r[f"{m}_summary"].strip() for m in MODELS}) == 3]
chosen = sorted(rng.sample(pool, N))

sheet, key = [], []
for n, i in enumerate(chosen, 1):
    r = rows[i]
    order = MODELS[:]
    rng.shuffle(order)                       # random A/B/C order, hides the model
    sheet.append({
        "id": n,
        "post": r["prompt"].strip(),
        "summary_A": r[f"{order[0]}_summary"].strip(),
        "summary_B": r[f"{order[1]}_summary"].strip(),
        "summary_C": r[f"{order[2]}_summary"].strip(),
        "member1_pick": "",
        "member2_pick": "",
    })
    key.append({"id": n, "A": order[0], "B": order[1], "C": order[2]})

# the sheet for the members (no model names)
with open("results/human_eval.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(sheet[0]))
    w.writeheader()
    w.writerows(sheet)

# which model wrote A/B/C (keep hidden until both members have picked)
with open("results/human_eval_key.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(key[0]))
    w.writeheader()
    w.writerows(key)

print(f"Saved results/human_eval.csv and results/human_eval_key.csv ({N} posts, seed {SEED})")

Saved results/human_eval.csv and results/human_eval_key.csv (30 posts, seed 42)


In [2]:
"""Per-member pick counts per method + agreement between the two members."""
import csv
from collections import Counter

MODELS = ["sft", "ppo", "dpo"]
sheet = list(csv.DictReader(open("results/human_eval.csv", encoding="utf-8-sig")))
key = {r["id"]: r for r in csv.DictReader(open("results/human_eval_key.csv", encoding="utf-8"))}

def model_of(row, col):
    pick = row[col].strip().upper()
    return key[row["id"]][pick] if pick in ("A", "B", "C") else None

counts = {"member1_pick": Counter(), "member2_pick": Counter()}
agree = valid = 0
for row in sheet:
    a, b = model_of(row, "member1_pick"), model_of(row, "member2_pick")
    if a: counts["member1_pick"][a] += 1
    if b: counts["member2_pick"][b] += 1
    if a and b:
        valid += 1
        agree += (a == b)

print(f"{'':10}" + "".join(f"{m.upper():>6}" for m in MODELS))
for col, label in (("member1_pick", "Member 1"), ("member2_pick", "Member 2")):
    print(f"{label:10}" + "".join(f"{counts[col][m]:>6}" for m in MODELS))
print(f"\nAgreement: {agree}/{valid} posts = {agree / max(valid, 1):.1%}")

             SFT   PPO   DPO
Member 1      11    14     5
Member 2      13     9     8

Agreement: 15/30 posts = 50.0%
